# FORUM: runbook and artifact validation

This notebook is the reproducible entry point for the manifest-driven FORUM analysis. It can run the upstream stages when `RUN_COMPLETE_PIPELINE` is explicitly set to `True`; by default it only inspects the completed artifacts. The statistical implementation lives in `commentgap_analysis.forum_scores`, and publication figures are generated by `commentgap_analysis.ranking_algorithm_effects`.

The legacy FORUM notebooks (`legacy/03B_forum-table.ipynb`, `legacy/04C_policy-robustness-and-tradeoffs.ipynb`, `legacy/05A_forum-calc-figs.ipynb`, `legacy/05E_forum-dist-figs.ipynb`, `legacy/05F_beta-figs.ipynb`, and `legacy/05G_latex-beta-tables.ipynb`) are methodological references only. This notebook never reads their outputs.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
FORUM_ANALYSIS_ROOT = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis'
INFERENCE_ROOT = FORUM_ANALYSIS_ROOT / 'inference'
REPORTING_ROOT = FORUM_ANALYSIS_ROOT / 'reporting'
print(f'Repository: {REPO_ROOT}')
print(f'FORUM output root: {FORUM_ANALYSIS_ROOT}')

## Frozen rankers used by FORUM

These are the `draw1` winners frozen by notebook 08 using development cross-validation only. The complete candidate ranking is also shown so the selected model is visible in context.


In [ ]:
from commentgap_analysis.forum_scores import freeze_ranker_handoff

selected_path = FORUM_ANALYSIS_ROOT / 'ranker_handoff/selected_rankers.csv'
ranking_path = FORUM_ANALYSIS_ROOT / 'ranker_handoff/development_cv_rankings.csv'
DRAW_POLICIES = ('draw1',)
freeze_ranker_handoff(
    factorial_root=REPO_ROOT / 'model_output/selection_2025/factorial_rankers',
    regression_scores_path=REPO_ROOT / 'model_output/selection_2025/regression/all/test_scores_wide.parquet',
    output_root=FORUM_ANALYSIS_ROOT / 'ranker_handoff',
    draw_policies=DRAW_POLICIES,
)

if selected_path.exists():
    selected_rankers = pd.read_csv(selected_path)
    display(selected_rankers[['family', 'feature_set', 'variant_id', 'audience_column', 'editor_column', 'score_path']])
else:
    print(f'Ranker handoff not available yet: {selected_path}')

if ranking_path.exists():
    candidate_rankers = pd.read_csv(ranking_path)
    display(candidate_rankers.sort_values(['family', 'feature_set', 'development_cv_rank'])[[
        'family', 'feature_set', 'draw_policy', 'variant_id',
        'development_cv_rank', 'mean_macro_ndcg_at_k', 'sd_macro_ndcg_at_k',
        'min_fold_macro_ndcg_at_k'
    ]])
else:
    print(f'Candidate ranking not available yet: {ranking_path}')


## Run missing stages, otherwise read existing artifacts

The cells below are the notebook execution path for the complete analysis. Each stage is checked independently and is run only when its expected output is absent or stale. The canonical outcomes are aggregate AQuA, article similarity, maximum toxicity, participant incumbency, prior audience reception, five-nearest-neighbour comment novelty, separate positive and negative sentiment, and raw CTTR and raw SMOG-DE. No alternative toxicity, novelty, semantic-atypicality, or individual-AQuA outcome variants are scored. Because this outcome revision requires rebuilding semantic novelty, set `COMMENTGAP_EMBEDDING_STORE` (or assign `EMBEDDING_STORE`) to the completed frozen BGE store before executing. Scoring uses all CPUs available to the process by default; set `COMMENTGAP_FORUM_WORKERS` to override this.


In [ ]:
from commentgap_analysis.forum_scores import (
    DEFAULT_WORKERS, FORUM_ANALYSIS_OUTCOMES, run_forum_analysis_pipeline
)

RUN_MISSING_STAGES = True
EMBEDDING_STORE = REPO_ROOT / (
    "model_output/selection_2025/embeddings/"
    "model=BAAI__bge-m3--d790e737/"
    "build=de5b3016fb2f-010a7cc75a88"
)
SCORING_WORKERS = int(os.getenv('COMMENTGAP_FORUM_WORKERS', str(DEFAULT_WORKERS)))

if RUN_MISSING_STAGES:
    pipeline = run_forum_analysis_pipeline(
        repo_root=REPO_ROOT,
        embedding_store=EMBEDDING_STORE,
        min_comments=11,
        progress_every_stories=10,
        outcomes=FORUM_ANALYSIS_OUTCOMES,
        workers=SCORING_WORKERS,
    )
    display(pd.DataFrame([
        {'stage': stage, 'status': status}
        for stage, status in pipeline['stage_status'].items()
    ]))
else:
    print('RUN_MISSING_STAGES=False: read-only validation mode.')


## Check the completed handoff

The expected primary sample is the held-out `paper2_test` partition with at least 11 comments. The large-thread sensitivity is recorded in the analysis table rather than silently changing the primary sample.

In [ ]:
expected = {
    'ranker handoff': FORUM_ANALYSIS_ROOT / 'ranker_handoff/ranker_handoff_manifest.json',
    'analysis comments': FORUM_ANALYSIS_ROOT / 'analysis_comments.parquet',
    'policy scores': FORUM_ANALYSIS_ROOT / 'policy_scores/policy_scores.parquet',
    'inference manifest': INFERENCE_ROOT / 'inference_manifest.json',
    'reporting manifest': REPORTING_ROOT / 'reporting_manifest.json',
}
status = pd.DataFrame([{'artifact': label, 'path': str(path), 'exists': path.exists()} for label, path in expected.items()])
display(status)
missing = status.loc[~status['exists'], 'artifact'].tolist()
if missing:
    print('Missing artifacts:', ', '.join(missing))
else:
    print('All expected FORUM stages are present.')

In [ ]:
def read_json(path):
    return json.loads(path.read_text())

manifests = {}
for label, path in expected.items():
    if path.suffix == '.json' and path.exists():
        manifests[label] = read_json(path)

if 'ranker handoff' in manifests:
    handoff = manifests['ranker handoff']
    print('Selected rankers:', len(handoff.get('selected_rankers', [])))
    print('Selection source:', handoff.get('selection_source'))
if 'inference manifest' in manifests:
    inference = manifests['inference manifest']
    print('Bootstrap draws:', inference.get('bootstrap_draws'))
    print('Resampling unit:', inference.get('paired_resampling_unit'))
if 'reporting manifest' in manifests:
    print('Report files:', len(manifests['reporting manifest'].get('outputs', {})))

In [ ]:
if expected['analysis comments'].exists() and expected['policy scores'].exists():
    comments = pd.read_parquet(expected['analysis comments'])
    scores = pd.read_parquet(expected['policy scores'])
    summary = pd.DataFrame({
        'quantity': ['comments', 'discussions', 'policy definitions', 'outcomes', 'depths', 'score rows'],
        'value': [
            len(comments), comments['story_id'].nunique(),
            scores[['policy_id', 'ordering', 'reply_mode', 'pinned']].drop_duplicates().shape[0],
            scores['outcome'].nunique(), scores['depth'].nunique(), len(scores),
        ],
    })
    display(summary)
    display(scores.groupby(['ordering', 'reply_mode', 'pinned'], dropna=False).size().to_frame('rows'))